## Descripción del proyecto

Este proyecto tiene el propósito de estudiar los datos de la empresa online Ice cuyo giro es la venta de videojuegos, con alcance en todo el mundo, para identificar patrones que ayuden a determinar si una próxima entrega tendría éxito o no con el fin de planificar estrategicamente campañas publicitarias.

## Importando librerías, cargando DataFrame y revisión general

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
from scipy import stats as st

In [2]:
df_games = pd.read_csv('./datasets/games.csv')

In [3]:
# Revisando la estructura general del DataFrame (no. de columnas, nombres de columnas, cantidad de filas, cuenta de datos no nulos y tipo de dato de las columnas)
df_games.info()

<class 'pandas.DataFrame'>
RangeIndex: 16715 entries, 0 to 16714
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16713 non-null  str    
 1   Platform         16715 non-null  str    
 2   Year_of_Release  16446 non-null  float64
 3   Genre            16713 non-null  str    
 4   NA_sales         16715 non-null  float64
 5   EU_sales         16715 non-null  float64
 6   JP_sales         16715 non-null  float64
 7   Other_sales      16715 non-null  float64
 8   Critic_Score     8137 non-null   float64
 9   User_Score       10014 non-null  str    
 10  Rating           9949 non-null   str    
dtypes: float64(6), str(5)
memory usage: 1.4 MB


En esta primera revisión con `info()`, se identificaron las siguientes observaciones sobre la estructura inicial del DataFrame:

- Se cuenta con 11 columnas donde el nombre de estas utilizan un formato (ej. `Year_of_Release`, `Critic_Score`, `User_Score`, etc.) que dificultaría la escritura para hacer referencia a ellos cuando se necesiten.
- Hay 6 columnas que cuentan con valores nulos y 5 que cuentan con valores en todas sus filas.

In [4]:
# Revisando la información de los primeros 5 registros
df_games.head()

,Name,Platform,Year_of_Release,Genre,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score,User_Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [5]:
print(f'Valores duplicados: {df_games.duplicated().sum()}\n')
print(f'Valores nulos\n{df_games.isna().sum()}')

Valores duplicados: 0

Valores nulos
Name                  2
Platform              0
Year_of_Release     269
Genre                 2
NA_sales              0
EU_sales              0
JP_sales              0
Other_sales           0
Critic_Score       8578
User_Score         6701
Rating             6766
dtype: int64


Al extraer los primeros registros puede observarse lo siguiente:
- El tipo de dato para la columna `Year_of_Release` es de tipo flotante cuando lo más indicado para este caso podría ser `object`
- Los valores encontrados en `NA_Sales`, `EU_Sales`, `JP_Sales` y `Other_sales` son de tipo flotante indicando la cantidad de ventas en millones de dólares (`USD`).
- La puntuación de la crítica (`Critic_Score`) se evalúa sobre 100 y es de tipo `float`.
- La puntuación del público (`User_Score`) se evalúa sobre 10 y es de tipo `object`.
- Aunque `Critic_Score` y `User_Score` muestren sus valores con punto decimal, ambas columnas tienen diferentes tipos de datos.
- La columna `Rating` almacena la clasificación de edad basada en la ESRB.


In [6]:
# Revisando una muestra de 10 registros aleatorios
df_games.sample(10)

,Name,Platform,Year_of_Release,Genre,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score,User_Score,Rating
11732,Minna no Joushiki Ryoku TV,Wii,2008.0,Misc,0.00,0.00,0.08,0.00,NaN,NaN,NaN
13523,Operation Flashpoint: Elite,XB,2005.0,Shooter,0.03,0.01,0.00,0.00,64.0,8.3,T
9682,Galidor: Defenders of the Outer Dimension,GBA,2002.0,Action,0.09,0.03,0.00,0.00,53.0,tbd,E
3623,Metal Gear Solid 3: Subsistence,PS2,2005.0,Action,0.34,0.01,0.15,0.06,94.0,9.3,M
13668,Puyo Pop Fever,PSP,2004.0,Puzzle,0.00,0.00,0.04,0.00,68.0,NaN,NaN
4059,Dragon Ball Z: Burst Limit,X360,2008.0,Fighting,0.24,0.18,0.03,0.05,72.0,7.6,T
6582,Secret Weapons Over Normandy,XB,2003.0,Simulation,0.19,0.05,0.00,0.01,77.0,8.8,T
5512,The X-Factor,Wii,2010.0,Misc,0.00,0.28,0.00,0.04,NaN,NaN,NaN
10789,Pro Yakyuu Famista DS,DS,2007.0,Sports,0.00,0.00,0.10,0.00,NaN,NaN,NaN
10179,Brave Story: New Traveler (US sales),PSP,2006.0,Role-Playing,0.11,0.00,0.00,0.00,NaN,NaN,NaN


Tomándo muestras aleatorias se detectó lo siguiente:
- La ausencia de ventas de un título para cualquiera de las regiones disponibles se registra con `0.0`.
- Hay valores ausentes en `Critic_Score`, `User_Score` y `Rating`.

### Resumen de las observaciones

Después de varias revisiones para la fase inicial de observación de la estructura del DataFrame se puede notar que las columnas no tienen un formato apropiado para las fases de análisis posteriores a la preparación de los datos. También, varias de las columnas necesitan convertirse a un tipo de dato más apropiado y hay tanto valores ausentes o valores posicionales temporales (ej. `tbd`) que se revisará más adelante la forma más apropiada de manejarlos en base a los requerimientos de análisis.

## Preparación de los datos

### Cambiando nombre de las columnas

In [7]:
# Convirtiendo nombres de las columnas en minúscula
df_games.columns = df_games.columns.str.lower()

# Cambiando nombre de la columna `year_of_release`
df_games = df_games.rename(columns= {'year_of_release': 'release_year'})

# Verificando los nuevos nombres de columna
print(df_games.columns.tolist())

['name', 'platform', 'release_year', 'genre', 'na_sales', 'eu_sales', 'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating']


### Tratando tipos de dato y valores nulos

#### `genre`

In [8]:
df_games['genre'] = df_games['genre'].fillna('Misc')

Revisando cuenta de valores nulos detecté que habían 2 registros vacíos para esta columna. Al existir un valor de `Misc` para juegos que no tienen una categoría específica, decidí rellenar los valores nulos con esta misma para incluirlos. Cabe mencionar que estos mismos registros corresponden también a los registros de `name` con valores nulos, pero esta acción de agregarlos al género `Misc` ayuda a agrupar esos registros con valores importantes en sus demás columnas.

#### `release_year`

In [9]:
# Convirtiendo tipo de dato para la columna de `release_year`
df_games['release_year'] = df_games['release_year'].astype('Int64').fillna(pd.NA)

El tipo de dato para `release_year` se cambió a `Int64` ya que estos eran `float64` siendo que esto no es lo más adecuado porque los años no se representan así. También, decidí llenar con `NA` los registros con valores nulos para no tomarlos en cuenta en los próximos análisis.

#### `critic_score`

In [10]:
df_games['critic_score'] = (
    df_games['critic_score']
    .astype('Int64')
    .fillna(pd.NA)
)

Se convirtió de `float64` a `Float64` ya que este último puede manejar al mismo tiempo valores flotantes como `NA`. Esto decidí hacerlo de esta manera porque al tratarse de puntuaciones o calificación, si se hacen analisis en base a región, plataforma o año de lanzamiento, esto podría afectar negativamente la calificación general que resulte de las operaciones si estos títulos con calificación `0` se incluyeran. Es mejor dejarlos fuera o ignorarlos.

#### `user_score`

In [11]:
df_games['user_score'] = (
    df_games['user_score']
    .replace('tbd', pd.NA)
    .astype('Float64')
    .fillna(pd.NA)
)

Reemplacé `tbd` por `NA` porque ultimadamente estos valores no podrían considerarse para un análisis sobre `user_score` debido a que representan una ausencia de datos. También, consideré mejor no usar la puntuación sobre 10 calculada desde `critic_score` porque se crearían opiniones falsas de los usuarios donde claramente es complicado que ambas puntuaciones coincidan.

#### `rating`

In [12]:
# Definiendo función para marcar los registros previo a 1994 (antes de la creación de la ESRB)
def pre_esrb(row):
    
    if (
        pd.notna(row['release_year'])
        and row['release_year'] < 1994
        and pd.isna(row['rating'])
        ):
        return 'Pre-ESRB'
    
    return row['rating']
        
    
df_games['rating'] = df_games.apply(pre_esrb, axis=1)

In [13]:
# Reemplazando antiguas califiaciones de la ESRB por la más moderna: E
# y sustituyendo RP (Rating Pending) por NA
standardize_ratings = {
    'K-A': 'E',
    'EC': 'E',
    'RP': np.nan
}

df_games['rating'] = df_games['rating'].replace(standardize_ratings)

# Rellenando valores nulos con la etiqueta 'Unrated'
df_games['rating'] = df_games['rating'].fillna('Unrated')

Para la columna `rating` apliqué una función (`pre_esrb`) que clasifica los juegos anteriores a 1994 (previo al año de la fundación de la ESRB) con la etiqueta `Pre-ESRB`. Esto para identificar fácilmente los registros que habrían sido nulos pero tienen una justificación para no tener una.

También, al revisar las filas me encontré con algunos registros que tenían asignado la clasificación de `K-A` que era la usada anteriormente a `E` (la que a día de hoy sigue usándose); se encontró también `EC` que fue absorvida por `E` debido a la poca utilización de esta por la ESRB.

Finalmente, me pareció adecuado cambiar la clasificación o placeholder de `RP` a `Unrated` ya que ultimadamente denota una ausencia de clasificación. Los valores nulos para los registros faltantes decidí también aplicar la misma etiqueta de `Unrated`.

### Agregando columna de ventas totales (todas las regiones)

In [ ]:
sales_columns = ['na_sales', 'eu_sales', 'jp_sales', 'other_sales']

df_games['total_sales'] = df_games[sales_columns].sum(axis=1).round(2)